In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%cd "/content/drive/MyDrive/TCC2"

In [ ]:
!ls -la
!ls -la "resnet50_experimento"
!ls -la "tcc_fotos"

In [ ]:
!pip install -q -r "resnet50_experimento/requirements.txt"

In [ ]:
!python "resnet50_experimento/00_tratar_imagens.py"

In [ ]:
!ls -la "tcc_fotos/dataset_tratado"

In [ ]:
!python "resnet50_experimento/01_preparar_dataset.py" --overwrite

In [ ]:
!ls -la "resnet50_experimento/resultados/dataset_resnet50"
!ls -la "resnet50_experimento/resultados/dataset_resnet50/train"
!ls -la "resnet50_experimento/resultados/dataset_resnet50/val"
!ls -la "resnet50_experimento/resultados/dataset_resnet50/test"

In [ ]:
!python "resnet50_experimento/02_treinar_resnet50.py"

In [ ]:
!python "resnet50_experimento/03_avaliar_resnet50.py"

In [ ]:
!ls -la "resnet50_experimento/resultados/resultados_tcc"

In [ ]:
!python "resnet50_experimento/04_gerar_figuras_tcc.py"

Testar modelo em imagens aleatórias

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from google.colab import files
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

# Caminho do melhor modelo treinado
caminho_modelo = "/content/drive/MyDrive/TCC2/resnet50_experimento/resultados/resultados_tcc/best_resnet50.keras"

print("Carregando o modelo...")
modelo = tf.keras.models.load_model(caminho_modelo)
print("Modelo carregado com sucesso!\n")


def crop_black_borders(img):
    """Remove bordas pretas da imagem laparoscópica."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        return img[y:y+h, x:x+w]

    return img


def remove_specular_reflections(img):
    """Reduz reflexos brancos usando inpainting."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 235, 255, cv2.THRESH_BINARY)

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)

    return cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)


def apply_clahe(img):
    """Aplica CLAHE para melhorar contraste."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)


def tratar_imagem(img_bgr):
    """Aplica o mesmo pipeline usado no experimento."""
    img_bgr = crop_black_borders(img_bgr)
    img_bgr = remove_specular_reflections(img_bgr)
    img_bgr = apply_clahe(img_bgr)
    return img_bgr


print("Faça o upload de uma ou mais imagens laparoscópicas para testar:")
uploaded = files.upload()

for arquivo in uploaded.keys():
    # Lê imagem crua em BGR
    img_original_bgr = cv2.imread(arquivo)

    if img_original_bgr is None:
        print(f"Não foi possível ler a imagem: {arquivo}")
        continue

    # Trata a imagem
    img_tratada_bgr = tratar_imagem(img_original_bgr)

    # Converte para RGB apenas para exibir e classificar
    img_original_rgb = cv2.cvtColor(img_original_bgr, cv2.COLOR_BGR2RGB)
    img_tratada_rgb = cv2.cvtColor(img_tratada_bgr, cv2.COLOR_BGR2RGB)

    # Mostra antes e depois
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img_original_rgb)
    plt.title("Imagem original")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(img_tratada_rgb)
    plt.title("Imagem tratada")
    plt.axis("off")

    plt.show()

    # Redimensiona para entrada da ResNet50
    img_resized = cv2.resize(img_tratada_rgb, (224, 224))

    # Converte para array e adiciona dimensão do lote
    img_array = image.img_to_array(img_resized)
    img_array = np.expand_dims(img_array, axis=0)

    # Predição
    probabilidade = float(modelo.predict(img_array, verbose=0)[0][0])

    if probabilidade >= 0.5:
        classe = "Com endometriose"
    else:
        classe = "Sem endometriose"

    print(f"Arquivo: {arquivo}")
    print(f"Classificação do modelo: {classe}")
    print(f"Probabilidade estimada para endometriose: {probabilidade:.2%}")
    print("-" * 60)